In [1]:
from contextlib import nullcontext

import torch
from tqdm.autonotebook import tqdm

from src.data.gaussian_mixture.data import MultimodalGaussianDataConfig
from src.deterministic_chart_transport.constraint import (
    ChartPretrainConfig,
    IntegratedChartConstraintConfig,
    LatentNormAnchorConfig,
    LatentScaleAnchorConfig,
    ReconstructionConfig,
)
from src.deterministic_chart_transport.critic import CriticLossConfig
from src.deterministic_chart_transport.model import ChartTransportModelConfig
from src.deterministic_chart_transport.study import (
    DeterministicChartTransportStudyConfig,
    DeterministicChartTransportStudyState,
)
from src.deterministic_chart_transport.transport import (
    DeterministicChartTransportLossConfig,
)
from src.model.mlp import StackedResidualMLPConfig
from src.model.time_conditioning import TimeConditioningConfig
from src.monitoring.utils import fieldplot, save_go_detached, scatterplot
from src.priors.anchored import AnchoredGaussianScaleMixturePriorConfig

device = torch.device('cuda:0')
artifact_root = 'artifacts/deterministic_chart_transport'
num_modes = 8
batch_size = 2048
ambient_dimension = 3
latent_dimension = ambient_dimension
hidden_dim = 512
hidden_dims = [hidden_dim] * 4


def get_device_context():
    if device.type == 'cuda':
        return torch.cuda.device(device)
    return nullcontext()


def get_autocast_context():
    if device.type == 'cuda':
        return torch.autocast(device_type=device.type, dtype=torch.bfloat16)
    return nullcontext()


/tmp/ipykernel_969881/3789874052.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
mc = ChartTransportModelConfig(
    encoder=StackedResidualMLPConfig.initialize(
        layer_dims=[ambient_dimension] + hidden_dims + [latent_dimension]
    ),
    decoder=StackedResidualMLPConfig.initialize(
        layer_dims=[latent_dimension] + hidden_dims + [ambient_dimension]
    ),
    critic=StackedResidualMLPConfig.initialize(
        layer_dims=[latent_dimension] + hidden_dims + [latent_dimension],
        time_conditioning_config=TimeConditioningConfig(
            min_t_lambda=1e-3,
            max_t_lambda=1.0,
            condition_dim=hidden_dim,
        ),
    ),
    chart_lr=3e-4,
    critic_lr=3e-4,
    grad_clip_norm=2.0,
)

data_reconstruction_config = ReconstructionConfig(
    huber_delta=10.0,
    weight=50.0,
)
pretrain_config = ChartPretrainConfig(
    data_reconstruction=data_reconstruction_config,
    prior_roundtrip=ReconstructionConfig(
        huber_delta=10.0,
        weight=1.0,
    ),
    anchor_config=LatentNormAnchorConfig(latent_norm_weight=1e-3),
)
transport_config = DeterministicChartTransportLossConfig(
    transport_step_multiplier=0.05,
    transport_step_cap=0.05,
    num_time_samples=5,
    t_range=(0.01, 0.99),
    antipodal_estimate=True,
    decoder_huber_delta=5.0,
    encoder_transport_weight=10.0,
    decoder_transport_weight=10.0,
)
integrated_constraint_config = IntegratedChartConstraintConfig(
    data_reconstruction=data_reconstruction_config,
    prior_roundtrip=ReconstructionConfig(
        huber_delta=10.0,
        weight=1.0,
    ),
    data_latent_anchor=LatentScaleAnchorConfig(
        latent_norm_weight=1.0,
        latent_zero_mean_weight=1.0,
        target_norm_per_dimension=1.0,
    ),
)

study_config = DeterministicChartTransportStudyConfig(
    data=MultimodalGaussianDataConfig.initialize(
        num_modes=num_modes,
        mode_std=0.05,
        ambient_dimension=ambient_dimension,
        offset=0.0,
    ),
    prior=AnchoredGaussianScaleMixturePriorConfig(
        latent_shape=[latent_dimension],
        precision=4.0,
    ),
    model=mc,
    pretrain=pretrain_config,
    critic=CriticLossConfig(
        huber_delta=10.0,
        weight=1.0,
        t_min=0.005,
        t_max=0.99,
    ),
    transport=transport_config,
    integrated_constraint=integrated_constraint_config,
)
display(study_config.visualize())
state = DeterministicChartTransportStudyState.initialize(
    config=study_config,
    device=device,
)


### Monitoring

In [3]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

visualize_batch_size_per_mode = 512
canonical_samples = {
    str(j): study_config.data.sample_class(
        mode_id=j,
        batch_size=visualize_batch_size_per_mode,
    ).to(device)
    for j in range(num_modes)
}
canonical_prior = state.prior_config.sample(
    batch_size=visualize_batch_size_per_mode,
).to(device)


def monitor(state):
    with torch.no_grad():
        model_sample = state.decode(canonical_prior)
        scatter_data_latent = {
            k: state.encode(data=v)
            for k, v in canonical_samples.items()
        }
        data_transport_field = {
            k: state.config.transport._estimate_transport_field(
                state,
                data_latent=v,
            )
            for k, v in scatter_data_latent.items()
        }
    conditioned_data_transport_field = {
        k: state.config.transport._scale_clip_transport_field(v)
        for k, v in data_transport_field.items()
    }

    data_scatter = scatterplot(scatter_data_latent)
    data_field = fieldplot(base=scatter_data_latent, field=data_transport_field)
    data_conditioned_field = fieldplot(
        base=scatter_data_latent,
        field=conditioned_data_transport_field,
    )
    model_scatter = scatterplot(canonical_samples | {'model': model_sample})

    combo = make_subplots(
        rows=2,
        cols=2,
        specs=[
            [{'type': 'scene'}, {'type': 'scene'}],
            [{'type': 'scene'}, {'type': 'scene'}],
        ],
    )
    for j, fig in enumerate(
        [
            data_scatter,
            data_field,
            data_conditioned_field,
            model_scatter,
        ]
    ):
        for tr in fig.data:
            combo.add_trace(tr, row=j // 2 + 1, col=j % 2 + 1)
    return combo


### Chart and critic pretrain

In [4]:
from pathlib import Path

Path(artifact_root).mkdir(parents=True, exist_ok=True)

pbar = tqdm(range(1_000))
for p in pbar:
    data = state.config.data.sample_unconditional(batch_size=batch_size).to(device)

    with get_device_context(), get_autocast_context():
        loss = state.compute_chart_pretrain_loss(data=data)
        total_loss = loss.sum()

    total_loss.backward()
    state.step_and_zero_grad()
    pbar.set_postfix({'loss': total_loss.item()})

pbar = tqdm(range(1_000))
for p in pbar:
    data = state.config.data.sample_unconditional(batch_size=batch_size).to(device)

    with get_device_context(), get_autocast_context():
        loss = state.compute_critic_only_loss(data=data)
        total_loss = loss.sum()

    total_loss.backward()
    state.step_and_zero_grad()
    pbar.set_postfix({'loss': total_loss.item()})

torch.save(state, f'{artifact_root}/simple_state.pkl')
monitor(state)


  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

In [5]:
state = torch.load(
    f'{artifact_root}/simple_state.pkl',
    map_location=device,
    weights_only=False,
)


## Integrated transport

In [ ]:
pbar = tqdm(range(20_000))

transport_chart_every_n_steps = 5
for p in pbar:
    data = state.config.data.sample_unconditional(batch_size=batch_size).to(device)

    with get_device_context(), get_autocast_context():
        losses = state.compute_integrated_loss(
            data=data,
            compute_transport_loss=(p % transport_chart_every_n_steps == 0),
        )
        total_loss = losses.sum()

    total_loss.backward()
    state.step_and_zero_grad()

    pbar.set_postfix({'loss': total_loss.item()})
    if p % transport_chart_every_n_steps == 0:
        if p % 200 == 0:
            fig = monitor(state)
            save_go_detached(
                fig,
                folder=artifact_root,
                name=str(p // transport_chart_every_n_steps),
            )

torch.save(state, f'{artifact_root}/simple_state.pkl')


  0%|          | 0/20000 [00:00<?, ?it/s]

: 